### Load the processed datasets

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

labeled_data = pd.read_csv(
    "../data/processed_data/labeled_data.csv"
)

prediction_data = pd.read_csv(
    "../data/processed_data/prediction_data.csv"
)

In [2]:
print("Labeled data shape:", labeled_data.shape)
print("Prediction data shape:", prediction_data.shape)

Labeled data shape: (198, 15)
Prediction data shape: (34, 14)


### Separate features and target`

In [3]:

X = labeled_data.drop(
    columns=["fault_status", "transformer_id"]
)

y = labeled_data["fault_status"].map({
    "Normal": 0,
    "Fault": 1
})

In [4]:
print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
print("\nTarget distribution:")
print(y.value_counts())

X shape: (198, 13)
y shape: (198,)


,install_year,rated_kva,cooling_type,zone,gas_mean,gas_max,gas_std,oil_temp_mean,oil_temp_max,oil_temp_std,load_mean,load_max,load_std
0,2012,250,ONAF,Zone 3,143.895994,147.817787,3.216604,63.097518,67.064311,4.360108,94.450842,104.927908,8.915228
1,2019,500,ONAN,Zone 3,100.991994,123.367978,15.073843,52.544203,65.398255,6.346153,65.642144,82.275383,8.850842
2,2009,500,OFAF,Zone 1,103.183072,122.118616,17.408723,57.813396,67.197059,5.172855,71.619793,82.604979,10.675000
3,2008,100,ONAF,Zone 1,125.595390,151.866470,19.034478,42.952912,49.972436,5.498816,50.071346,70.107299,9.690446
4,2006,250,ONAN,Zone 2,115.326868,202.066251,43.176527,51.196584,53.452318,1.964975,64.048775,73.754073,8.496023



Target distribution:
0    149
1     49
Name: fault_status, dtype: int64


### Train/Test Split

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [6]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

X_train: (158, 13)
X_test : (40, 13)
y_train: (158,)
y_test : (40,)

Training target distribution:
0    119
1     39
Name: fault_status, dtype: int64

Test target distribution:
0    30
1    10
Name: fault_status, dtype: int64


### Prepare Numerical & Categorical Features

In [7]:
# Define the feature groups
categorical_features = [
    "cooling_type",
    "zone"
]

numerical_features = [
    "install_year",
    "rated_kva",
    "gas_mean",
    "gas_max",
    "gas_std",
    "oil_temp_mean",
    "oil_temp_max",
    "oil_temp_std",
    "load_mean",
    "load_max",
    "load_std"
]


In [8]:
print("Categorical features:", categorical_features)
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features:", numerical_features)
print("Number of numerical features:", len(numerical_features))

Categorical features: ['cooling_type', 'zone']
Number of categorical features: 2

Numerical features: ['install_year', 'rated_kva', 'gas_mean', 'gas_max', 'gas_std', 'oil_temp_mean', 'oil_temp_max', 'oil_temp_std', 'load_mean', 'load_max', 'load_std']
Number of numerical features: 11


In [9]:
# Build the Preprocessor

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [10]:
# Fit and Transform the Data

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [11]:
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape :", X_test_processed.shape)

X_train_processed shape: (158, 18)
X_test_processed shape : (40, 18)


In [12]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print("\nProcessed feature names:")

for feature in feature_names:
    print(feature)

Number of processed features: 18

Processed feature names:
num__install_year
num__rated_kva
num__gas_mean
num__gas_max
num__gas_std
num__oil_temp_mean
num__oil_temp_max
num__oil_temp_std
num__load_mean
num__load_max
num__load_std
cat__cooling_type_OFAF
cat__cooling_type_ONAF
cat__cooling_type_ONAN
cat__cooling_type_Unknown
cat__zone_Zone 1
cat__zone_Zone 2
cat__zone_Zone 3


### Build Model

In [13]:
# Train an initial Logistic Regression model as a model candidate.

In [14]:
# Import and create the model
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    random_state=42
)


In [15]:
logistic_model.fit(
    X_train_processed,
    y_train
)

LogisticRegression(random_state=42)

In [16]:
# Make Predictions on the Test Set

y_pred = logistic_model.predict(X_test_processed)

In [17]:
print("Predictions:", y_pred)

Predictions: [0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0]


In [18]:
print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts())


Prediction distribution:
0    38
1     2
dtype: int64


In [19]:
# Evaluate the Baseline Model

from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Fault"]
))

              precision    recall  f1-score   support

      Normal       0.76      0.97      0.85        30
       Fault       0.50      0.10      0.17        10

    accuracy                           0.75        40
   macro avg       0.63      0.53      0.51        40
weighted avg       0.70      0.75      0.68        40



In [20]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[29  1]
 [ 9  1]]


### Address Class Imbalance

In [21]:
# Train a balanced Logistic Regression

logistic_balanced = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

In [22]:
logistic_balanced.fit(
    X_train_processed,
    y_train
)

LogisticRegression(class_weight='balanced', random_state=42)

In [23]:
y_pred_balanced = logistic_balanced.predict(X_test_processed)

In [24]:
print("Predictions:", y_pred_balanced)

Predictions: [1 1 0 0 0 1 1 0 0 1 1 1 1 0 1 0 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 1 1 1 0 0
 0 0 0]


In [25]:
print("\nPrediction distribution:")
print(pd.Series(y_pred_balanced).value_counts())


Prediction distribution:
0    24
1    16
dtype: int64


In [26]:
# Evaluate the Balanced Model

from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_balanced,
    target_names=["Normal", "Fault"]
))

              precision    recall  f1-score   support

      Normal       0.83      0.67      0.74        30
       Fault       0.38      0.60      0.46        10

    accuracy                           0.65        40
   macro avg       0.60      0.63      0.60        40
weighted avg       0.72      0.65      0.67        40



In [27]:
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

print("\nConfusion Matrix:")
print(cm_balanced)


Confusion Matrix:
[[20 10]
 [ 4  6]]


### Trying a different model

In [28]:
# Train Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=42
)

In [29]:
rf_model.fit(
    X_train_processed,
    y_train
)

RandomForestClassifier(random_state=42)

In [30]:
y_pred_rf = rf_model.predict(X_test_processed)

In [31]:
print("Predictions:", y_pred_rf)

Predictions: [0 0 0 0 0 0 1 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0
 0 0 0]


In [32]:
print("\nPrediction distribution:")
print(pd.Series(y_pred_rf).value_counts())


Prediction distribution:
0    34
1     6
dtype: int64


In [33]:
# Evaluate Random Forest

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=["Normal", "Fault"]
))

              precision    recall  f1-score   support

      Normal       0.82      0.93      0.87        30
       Fault       0.67      0.40      0.50        10

    accuracy                           0.80        40
   macro avg       0.75      0.67      0.69        40
weighted avg       0.78      0.80      0.78        40



In [34]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("\nConfusion Matrix:")
print(cm_rf)


Confusion Matrix:
[[28  2]
 [ 6  4]]


In [35]:
# Balanced Random Forest

rf_balanced = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

rf_balanced.fit(
    X_train_processed,
    y_train
)

y_pred_rf_balanced = rf_balanced.predict(X_test_processed)

In [36]:
print(classification_report(
    y_test,
    y_pred_rf_balanced,
    target_names=["Normal", "Fault"]
))

cm_rf_balanced = confusion_matrix(y_test, y_pred_rf_balanced)

print("\nConfusion Matrix:")
print(cm_rf_balanced)

              precision    recall  f1-score   support

      Normal       0.85      0.97      0.91        30
       Fault       0.83      0.50      0.62        10

    accuracy                           0.85        40
   macro avg       0.84      0.73      0.77        40
weighted avg       0.85      0.85      0.84        40


Confusion Matrix:
[[29  1]
 [ 5  5]]


### Cross-validation

In [37]:
# import Stratified K-Fold

from sklearn.model_selection import StratifiedKFold, cross_validate

In [38]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [39]:
# Create the two pipelines

from sklearn.pipeline import Pipeline

In [40]:
# Balanced Logistic:

logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        random_state=42
    ))
])

In [41]:
# Balanced Random Forest:

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

In [42]:
# evaluation metrics

from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0)
}

In [43]:
# Run CV for Balanced Logistic

logistic_cv = cross_validate(
    logistic_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring
)

In [44]:
# Balanced Random Forest:

rf_cv = cross_validate(
    rf_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring
)

In [45]:
cv_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Fault Precision",
        "Fault Recall",
        "Fault F1"
    ],
    "Balanced Logistic": [
        logistic_cv["test_accuracy"].mean(),
        logistic_cv["test_precision"].mean(),
        logistic_cv["test_recall"].mean(),
        logistic_cv["test_f1"].mean()
    ],
    "Balanced Random Forest": [
        rf_cv["test_accuracy"].mean(),
        rf_cv["test_precision"].mean(),
        rf_cv["test_recall"].mean(),
        rf_cv["test_f1"].mean()
    ]
})

cv_results.round(3)

,Metric,Balanced Logistic,Balanced Random Forest
0,Accuracy,0.651,0.748
1,Fault Precision,0.391,0.320
2,Fault Recall,0.613,0.189
3,Fault F1,0.472,0.231


In [46]:
# Based on the evidence we have now:
# Balanced Logistic Regression is currently our strongest candidate.

In [47]:
print("Balanced Logistic - Fold Results")
print("----------------------------------")

for i, (precision, recall, f1) in enumerate(
    zip(
        logistic_cv["test_precision"],
        logistic_cv["test_recall"],
        logistic_cv["test_f1"]
    ),
    start=1
):
    print(
        f"Fold {i}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

print("\nBalanced Random Forest - Fold Results")
print("--------------------------------------")

for i, (precision, recall, f1) in enumerate(
    zip(
        rf_cv["test_precision"],
        rf_cv["test_recall"],
        rf_cv["test_f1"]
    ),
    start=1
):
    print(
        f"Fold {i}: "
        f"Precision={precision:.3f}, "
        f"Recall={recall:.3f}, "
        f"F1={f1:.3f}"
    )

Balanced Logistic - Fold Results
----------------------------------
Fold 1: Precision=0.333, Recall=0.600, F1=0.429
Fold 2: Precision=0.467, Recall=0.700, F1=0.560
Fold 3: Precision=0.545, Recall=0.600, F1=0.571
Fold 4: Precision=0.294, Recall=0.500, F1=0.370
Fold 5: Precision=0.316, Recall=0.667, F1=0.429

Balanced Random Forest - Fold Results
--------------------------------------
Fold 1: Precision=0.000, Recall=0.000, F1=0.000
Fold 2: Precision=0.600, Recall=0.300, F1=0.400
Fold 3: Precision=0.000, Recall=0.000, F1=0.000
Fold 4: Precision=0.500, Recall=0.200, F1=0.286
Fold 5: Precision=0.500, Recall=0.444, F1=0.471


In [48]:
# threshold tuning



In [49]:
y_prob = logistic_balanced.predict_proba(X_test_processed)[:, 1]

print("Fault probabilities:")
print(y_prob)

Fault probabilities:
[0.51532189 0.54828198 0.34132897 0.35055288 0.47460128 0.60953758
 0.76085294 0.31253251 0.24861249 0.79767215 0.86514049 0.55719788
 0.74533664 0.48163045 0.70238313 0.27967886 0.36495062 0.29857088
 0.62095142 0.33561783 0.2173087  0.45538984 0.27805111 0.40862863
 0.71450208 0.29891468 0.62772857 0.14109832 0.69237114 0.46319083
 0.2052889  0.46791751 0.6992205  0.68469047 0.66602618 0.29390963
 0.4101239  0.2416225  0.43575646 0.23655934]


In [50]:
for threshold in [0.3, 0.4, 0.5, 0.6]:
    
    y_pred_threshold = (y_prob >= threshold).astype(int)
    
    print(f"\nThreshold: {threshold}")
    print(classification_report(
        y_test,
        y_pred_threshold,
        target_names=["Normal", "Fault"],
        zero_division=0
    ))


Threshold: 0.3
              precision    recall  f1-score   support

      Normal       0.91      0.33      0.49        30
       Fault       0.31      0.90      0.46        10

    accuracy                           0.48        40
   macro avg       0.61      0.62      0.47        40
weighted avg       0.76      0.47      0.48        40


Threshold: 0.4
              precision    recall  f1-score   support

      Normal       0.88      0.47      0.61        30
       Fault       0.33      0.80      0.47        10

    accuracy                           0.55        40
   macro avg       0.60      0.63      0.54        40
weighted avg       0.74      0.55      0.57        40


Threshold: 0.5
              precision    recall  f1-score   support

      Normal       0.83      0.67      0.74        30
       Fault       0.38      0.60      0.46        10

    accuracy                           0.65        40
   macro avg       0.60      0.63      0.60        40
weighted avg       0.72   

In [51]:
# Model: Balanced Logistic Regression 
# Primary metric: Fault recall / Fault F1
# Compare different probability thresholds to understand the precision-recall trade-off.

In [52]:
print("Balanced Logistic Regression - CV Stability")
print("---------------------------------------------")

for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = logistic_cv[f"test_{metric}"]
    
    print(
        f"{metric.capitalize():<10}: "
        f"Mean={scores.mean():.3f}, "
        f"Std={scores.std():.3f}"
    )

Balanced Logistic Regression - CV Stability
---------------------------------------------
Accuracy  : Mean=0.651, Std=0.083
Precision : Mean=0.391, Std=0.098
Recall    : Mean=0.613, Std=0.069
F1        : Mean=0.472, Std=0.080


### predict the 34 transformers

In [53]:
# Step 1 — Fit the final pipeline

# We already had created logistic_pipeline 

# Train final model on all labeled data

logistic_pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  ['install_year', 'rated_kva',
                                                   'gas_mean', 'gas_max',
                                                   'gas_std', 'oil_temp_mean',
                                                   'oil_temp_max',
                                                   'oil_temp_std', 'load_mean',
                                                   'load_max', 'load_std']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['cooling_type', 'zone'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', random_state=42))])

In [54]:
# Step 2 — Prepare the 34 prediction transformers

X_prediction = prediction_data.drop(
    columns=["transformer_id"]
)

print("Prediction data shape:", X_prediction.shape)

Prediction data shape: (34, 13)


In [55]:
# Step 3 — Predict

prediction_labels = logistic_pipeline.predict(X_prediction)

In [56]:
# Convert the numeric predictions back to meaningful labels:

prediction_status = pd.Series(
    prediction_labels
).map({
    0: "Normal",
    1: "Fault"
})

In [57]:
print("Prediction distribution:")
print(prediction_status.value_counts())

Prediction distribution:
Normal    21
Fault     13
dtype: int64


In [58]:
# Adding Fault probabilities

# Get probability of Fault for each prediction
prediction_probability = logistic_pipeline.predict_proba(
    X_prediction
)[:, 1]

In [59]:
final_predictions = pd.DataFrame({
    "transformer_id": prediction_data["transformer_id"],
    "predicted_fault_status": prediction_status,
    "fault_probability": prediction_probability
})

display(final_predictions)

,transformer_id,predicted_fault_status,fault_probability
0,TXR-0012,Normal,0.298418
1,TXR-0022,Fault,0.534337
2,TXR-0049,Fault,0.706366
3,TXR-0050,Normal,0.272061
4,TXR-0055,Fault,0.574839
5,TXR-0058,Normal,0.223042
6,TXR-0074,Normal,0.183323
7,TXR-0077,Fault,0.917984
8,TXR-0079,Normal,0.183953
9,TXR-0088,Normal,0.273609


In [60]:
final_predictions["fault_probability"] = (
    final_predictions["fault_probability"].round(3)
)

display(final_predictions)

,transformer_id,predicted_fault_status,fault_probability
0,TXR-0012,Normal,0.298
1,TXR-0022,Fault,0.534
2,TXR-0049,Fault,0.706
3,TXR-0050,Normal,0.272
4,TXR-0055,Fault,0.575
5,TXR-0058,Normal,0.223
6,TXR-0074,Normal,0.183
7,TXR-0077,Fault,0.918
8,TXR-0079,Normal,0.184
9,TXR-0088,Normal,0.274


In [61]:
print(
    "Fault probability range:",
    final_predictions["fault_probability"].min(),
    "to",
    final_predictions["fault_probability"].max()
)

Fault probability range: 0.119 to 0.918


### Final sanity checks

In [62]:
# Final prediction sanity checks

print("Number of predictions:", len(final_predictions))
print(
    "Unique transformer IDs:",
    final_predictions["transformer_id"].nunique()
)
print(
    "Missing predictions:",
    final_predictions["predicted_fault_status"].isna().sum()
)
print(
    "Duplicate transformer IDs:",
    final_predictions["transformer_id"].duplicated().sum()
)

print("\nPrediction distribution:")
print(final_predictions["predicted_fault_status"].value_counts())

print("\nProbability range:")
print(
    final_predictions["fault_probability"].min(),
    "to",
    final_predictions["fault_probability"].max()
)

Number of predictions: 34
Unique transformer IDs: 34
Missing predictions: 0
Duplicate transformer IDs: 0

Prediction distribution:
Normal    21
Fault     13
Name: predicted_fault_status, dtype: int64

Probability range:
0.119 to 0.918


In [63]:
# Saving the predictions:

final_predictions.to_csv(
    r"C:\Users\Nithin\Documents\DATA SCIENCE ACADEMY PROJECTS\Project\classification\data\processed_data\final_predictions.csv",
    index=False
)

In [64]:
final_predictions.head()


,transformer_id,predicted_fault_status,fault_probability
0,TXR-0012,Normal,0.298
1,TXR-0022,Fault,0.534
2,TXR-0049,Fault,0.706
3,TXR-0050,Normal,0.272
4,TXR-0055,Fault,0.575


In [65]:
print("Final prediction file saved successfully.")

Final prediction file saved successfully.


### Testing out new model

In [66]:
# Create the SVM model

from sklearn.svm import SVC

svm_model = SVC(
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_processed, y_train)

y_pred_svm = svm_model.predict(X_test_processed)

In [67]:
# Evaluate it :
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("SVM Classification Report")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Normal", "Fault"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

print("Accuracy:", accuracy_score(y_test, y_pred_svm))

SVM Classification Report
              precision    recall  f1-score   support

      Normal       0.84      0.53      0.65        30
       Fault       0.33      0.70      0.45        10

    accuracy                           0.57        40
   macro avg       0.59      0.62      0.55        40
weighted avg       0.71      0.57      0.60        40

Confusion Matrix:
[[16 14]
 [ 3  7]]
Accuracy: 0.575


In [68]:
# Create the SVM pipeline

svm_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVC(
        class_weight="balanced",
        random_state=42
    ))
])

In [69]:
# Run the same 5-fold CV

svm_cv = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=cv,
    scoring=scoring
)

In [70]:
svm_cv_results = {
    "Accuracy": svm_cv["test_accuracy"].mean(),
    "Precision": svm_cv["test_precision"].mean(),
    "Recall": svm_cv["test_recall"].mean(),
    "F1": svm_cv["test_f1"].mean()
}

print("SVM 5-Fold Cross-Validation Results")
print("-" * 45)

for metric, value in svm_cv_results.items():
    print(f"{metric}: {value:.3f}")

SVM 5-Fold Cross-Validation Results
---------------------------------------------
Accuracy: 0.611
Precision: 0.315
Recall: 0.476
F1: 0.371


### Final Model Comparison

The three classification models were compared using 5-fold Stratified Cross-Validation.

The main focus was on Fault Recall and Fault F1-score because the objective is to detect faulty transformers effectively.

In [71]:
# Final comparison of all classification models

model_comparison = pd.DataFrame({
    "Model": [
        "Balanced Logistic Regression",
        "Balanced Random Forest",
        "SVM"
    ],
    "CV Accuracy": [
        logistic_cv["test_accuracy"].mean(),
        rf_cv["test_accuracy"].mean(),
        svm_cv["test_accuracy"].mean()
    ],
    "Fault Precision": [
        logistic_cv["test_precision"].mean(),
        rf_cv["test_precision"].mean(),
        svm_cv["test_precision"].mean()
    ],
    "Fault Recall": [
        logistic_cv["test_recall"].mean(),
        rf_cv["test_recall"].mean(),
        svm_cv["test_recall"].mean()
    ],
    "Fault F1": [
        logistic_cv["test_f1"].mean(),
        rf_cv["test_f1"].mean(),
        svm_cv["test_f1"].mean()
    ]
})

model_comparison.round(3)

,Model,CV Accuracy,Fault Precision,Fault Recall,Fault F1
0,Balanced Logistic Regression,0.651,0.391,0.613,0.472
1,Balanced Random Forest,0.748,0.320,0.189,0.231
2,SVM,0.611,0.315,0.476,0.371


### Model Selection

Based on the 5-fold cross-validation results, Balanced Logistic Regression was retained as the final model.

Although Balanced Random Forest achieved higher overall accuracy, its Fault recall was considerably lower. Since detecting Fault transformers is the primary objective, Balanced Logistic Regression provided the best balance of Fault recall, precision, and F1-score.

SVM was also evaluated but did not outperform Balanced Logistic Regression.

#### Tuning Balanced Logistic Regression & comparing for better results


In [72]:
# Let's test 3 values

# Tune Logistic Regression using different C values

C_values = [0.1, 1, 10]

logistic_tuning_results = []

for c in C_values:

    logistic_pipeline_tuned = Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=c,
            class_weight="balanced",
            random_state=42
        ))
    ])

    cv_result = cross_validate(
        logistic_pipeline_tuned,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    logistic_tuning_results.append({
        "C": c,
        "CV Accuracy": cv_result["test_accuracy"].mean(),
        "Fault Precision": cv_result["test_precision"].mean(),
        "Fault Recall": cv_result["test_recall"].mean(),
        "Fault F1": cv_result["test_f1"].mean()
    })

logistic_tuning_results_df = pd.DataFrame(logistic_tuning_results)

logistic_tuning_results_df.round(3)

,C,CV Accuracy,Fault Precision,Fault Recall,Fault F1
0,0.1,0.656,0.378,0.576,0.453
1,1.0,0.651,0.391,0.613,0.472
2,10.0,0.676,0.409,0.596,0.479


In [73]:
# Among the tested values, `C=10` achieved the highest CV accuracy (0.676), Fault precision (0.409), 
# and Fault F1-score (0.479). Although `C=1` achieved slightly higher Fault recall (0.613 compared with 0.596), 
# `C=10` provided the best overall balance between precision and recall.

In [74]:
# Final tuned Logistic Regression model

logistic_tuned_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        C=10,
        class_weight="balanced",
        random_state=42
    ))
])

In [75]:
logistic_tuned_pipeline.fit(X_train, y_train)

y_pred_tuned = logistic_tuned_pipeline.predict(X_test)

print("Tuned Logistic Regression (C=10)")
print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=["Normal", "Fault"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("Accuracy:", accuracy_score(y_test, y_pred_tuned))

Tuned Logistic Regression (C=10)
              precision    recall  f1-score   support

      Normal       0.87      0.67      0.75        30
       Fault       0.41      0.70      0.52        10

    accuracy                           0.68        40
   macro avg       0.64      0.68      0.64        40
weighted avg       0.76      0.68      0.70        40

Confusion Matrix:
[[20 10]
 [ 3  7]]
Accuracy: 0.675


In [76]:
# Train the final tuned model on all labeled data

logistic_tuned_pipeline.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', MinMaxScaler(),
                                                  ['install_year', 'rated_kva',
                                                   'gas_mean', 'gas_max',
                                                   'gas_std', 'oil_temp_mean',
                                                   'oil_temp_max',
                                                   'oil_temp_std', 'load_mean',
                                                   'load_max', 'load_std']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['cooling_type', 'zone'])])),
                ('model',
                 LogisticRegression(C=10, class_weight='balanced',
                                    random_state=42))])

In [77]:
X_prediction = prediction_data.drop(columns=["transformer_id"])

prediction_labels = logistic_tuned_pipeline.predict(X_prediction)

prediction_status = pd.Series(prediction_labels).map({
    0: "Normal",
    1: "Fault"
})

prediction_probability = (
    logistic_tuned_pipeline.predict_proba(X_prediction)[:, 1]
)

final_predictions_tuned = pd.DataFrame({
    "transformer_id": prediction_data["transformer_id"],
    "predicted_fault_status": prediction_status,
    "fault_probability": prediction_probability.round(3)
})

final_predictions_tuned

,transformer_id,predicted_fault_status,fault_probability
0,TXR-0012,Normal,0.169
1,TXR-0022,Normal,0.482
2,TXR-0049,Fault,0.827
3,TXR-0050,Normal,0.170
4,TXR-0055,Fault,0.628
5,TXR-0058,Normal,0.153
6,TXR-0074,Normal,0.099
7,TXR-0077,Fault,0.983
8,TXR-0079,Normal,0.087
9,TXR-0088,Normal,0.160


In [78]:
print("Number of predictions:", len(final_predictions_tuned))
print("Unique transformer IDs:", final_predictions_tuned["transformer_id"].nunique())
print("Missing predictions:", final_predictions_tuned["predicted_fault_status"].isna().sum())
print("Duplicate transformer IDs:", final_predictions_tuned["transformer_id"].duplicated().sum())

print("\nPrediction distribution:")
print(final_predictions_tuned["predicted_fault_status"].value_counts())

print("\nProbability range:")
print(
    final_predictions_tuned["fault_probability"].min(),
    "to",
    final_predictions_tuned["fault_probability"].max()
)

Number of predictions: 34
Unique transformer IDs: 34
Missing predictions: 0
Duplicate transformer IDs: 0

Prediction distribution:
Normal    20
Fault     14
Name: predicted_fault_status, dtype: int64

Probability range:
0.042 to 0.983


In [79]:
final_predictions_tuned.to_csv(
    "../data/processed_data/final_predictions.csv",
    index=False
)

print("Final tuned prediction file saved successfully.")

Final tuned prediction file saved successfully.
